# M&A Buyer Likelihood Screener

Provide a target company profile and the screener ranks every acquirer in the
dataset by how closely their historical deal behaviour matches your target.

**Workflow:**
1. **Setup** — load the transaction dataset
2. **Target Profile** — describe the company you are evaluating
3. **Scoring Engine** — builds acquirer profiles and scores each dimension
4. **Feasibility Pipeline** — runs web-enriched LLM assessments and exports PDFs

| Scoring dimension | Weight | Logic |
|---|---|---|
| Sector fit        | 30 % | Share of acquirer's past deals in the target's sector |
| Deal size fit     | 25 % | Z-score proximity to acquirer's typical deal size |
| Geography fit     | 15 % | Share of acquirer's past deals in the target's region |
| EBITDA margin fit | 15 % | Proximity to acquirer's historical margin preference |
| Ownership fit     | 10 % | Share of acquirer's past deals with same ownership type |
| Recency           |  5 % | How recently the acquirer has been active |

Scores are 0–100 per dimension; the composite score is the weighted sum.

## Setup

Imports and data load. Only **closed** deals are used for historical pattern analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'sans-serif', 'font.size': 10})
sns.set(style='whitegrid', palette='muted')

df = pd.read_csv('ma_transactions_500.csv')
for col in ['deal_size_mm', 'ebitda_margin_pct', 'revenue_growth_pct',
            'ev_ebitda_multiple', 'days_to_close']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Keep only closed deals for historical pattern analysis
closed = df[df['outcome'] == 'Closed'].copy()
print(f"{len(df):,} total transactions | {len(closed):,} closed | "
      f"{closed['acquirer'].nunique()} unique acquirers")
closed.head(3)

500 total transactions | 406 closed | 97 unique acquirers


,transaction_id,target_company,acquirer,sector,sub_sector,deal_year,deal_quarter,deal_type,geography,financing_type,...,revenue_growth_pct,ev_ebitda_multiple,ev_revenue_multiple,synergy_pct_of_deal,outcome,strategic_rationale_tags,num_bidders,days_to_close,acquirer_type,target_ownership_pre
0,MA-2020-0001,Pinnacle Revenue Cycle,Phreesia,Health IT,Health IT,2020,Q4,Strategic Acquisition,Northeast,All Cash,...,13.4,18.3,5.20,15.1,Closed,Technology Capability|Geographic Expansion|Ver...,3,157.0,Strategic,Public
1,MA-2024-0002,Great Medical Associates,Hellman & Friedman,Physician Groups,Physician Groups,2024,Q3,Bolt-on Acquisition,Southeast,Leveraged,...,13.1,13.3,1.38,13.5,Closed,Geographic Expansion|Platform Build|Diversific...,5,108.0,Financial Sponsor,Public
2,MA-2018-0003,Select Health Data,TPG Capital,Health IT,Health IT,2018,Q1,Carve-out,Southeast,Leveraged,...,18.5,23.3,5.09,11.4,Closed,Technology Capability|Geographic Expansion|Hig...,2,404.0,Financial Sponsor,PE-Backed


## 1. Target Company Profile

Edit the `TARGET` dict in the cell below to describe the company you are
evaluating — **this is the only cell you need to change for a new target.**

- Financials accept a number, a qualitative descriptor
  (`"strong"` / `"above average"` / `"average"` / `"below average"` / `"weak"`),
  or `None` to exclude the dimension (weight is redistributed automatically).
- Weights must sum to `1.0`.

In [2]:
TARGET = {
    # ── Identity ──────────────────────────────────────────────────────────────
    "name": "Example Target Co.",

    # ── Business Profile ──────────────────────────────────────────────────────
    "sector":    "Health IT",          # must match a sector in the dataset
    "geography": "Northeast",          # must match a geography in the dataset
    "ownership": "Private",            # Public | Private | PE-Backed | Non-Profit

    # ── Deal Financials ───────────────────────────────────────────────────────
    # Each field accepts:
    #   - a number              e.g. 22.0
    #   - a qualitative string  "strong" | "above average" | "average" | "below average" | "weak"
    #       → resolves to the sector-specific percentile from the dataset
    #   - None / "NA"           → dimension excluded from scoring (weight redistributed)
    "deal_size_mm":      200.0,
    "ebitda_margin_pct": "strong",

    # ── Acquirer Type Preference ──────────────────────────────────────────────
    # Leave as None to consider all, or set to "Financial Sponsor" / "Strategic"
    "preferred_acquirer_type": None,

    # ── Scoring Weights (must sum to 1.0) ─────────────────────────────────────
    "weights": {
        "sector":        0.30,
        "deal_size":     0.25,
        "geography":     0.15,
        "ebitda_margin": 0.15,
        "ownership":     0.10,
        "recency":       0.05,
    }
}

assert abs(sum(TARGET['weights'].values()) - 1.0) < 1e-9, \
    f"Weights sum to {sum(TARGET['weights'].values()):.4f} — must equal 1.0"


### Resolver
Converts qualitative descriptors to sector-specific percentile values and
redistributes weight from any excluded dimensions.

In [3]:
_QUALITATIVE_PERCENTILES = {
    "strong":        0.75,
    "above average": 0.62,
    "average":       0.50,
    "below average": 0.38,
    "weak":          0.25,
}

# Maps profile financial fields to their scoring dimension name
_FINANCIAL_FIELDS = [
    ('ebitda_margin_pct', 'ebitda_margin'),
]

def resolve_target_financials(target, df):
    """
    - Qualitative strings → sector-specific percentile value
    - None / "NA" → field excluded; its weight redistributed to remaining dimensions
    Returns a resolved copy of target with numeric values and adjusted weights.
    """
    t = target.copy()
    w = t['weights'].copy()
    sector_data = df[df['sector'] == t['sector']]
    if sector_data.empty:
        sector_data = df

    na_dims = []

    for field, dim in _FINANCIAL_FIELDS:
        val = t.get(field)

        if val is None or (isinstance(val, str) and val.strip().lower() == 'na'):
            na_dims.append(dim)
            t[field] = None
            print(f"  {field}: unknown — excluding from scoring")
            continue

        if isinstance(val, str):
            key = val.lower().strip()
            if key not in _QUALITATIVE_PERCENTILES:
                raise ValueError(
                    f"Unknown descriptor '{val}'. "
                    f"Use: {list(_QUALITATIVE_PERCENTILES.keys())} or None"
                )
            pct = _QUALITATIVE_PERCENTILES[key]
            resolved = sector_data[field].dropna().quantile(pct)
            print(f"  '{val}' {field} for '{t['sector']}' → {resolved:.1f}% "
                  f"(p{pct*100:.0f} of sector)")
            t[field] = resolved

    # Redistribute freed weight proportionally across remaining active dimensions
    if na_dims:
        freed = sum(w[d] for d in na_dims if d in w)
        remaining = {d: v for d, v in w.items() if d not in na_dims}
        total_remaining = sum(remaining.values())
        for d in remaining:
            w[d] += freed * (w[d] / total_remaining)
        for d in na_dims:
            if d in w:
                w[d] = 0.0
        print(f"  Redistributed weights: " +
              ", ".join(f"{d}={v:.2f}" for d, v in w.items() if v > 0))

    t['weights'] = w
    t['_na_dims'] = na_dims
    return t


print(f"Target   : {TARGET['name']}")
print(f"Sector   : {TARGET['sector']}  |  Geography: {TARGET['geography']}  |  Ownership: {TARGET['ownership']}")
print(f"Size     : ${TARGET['deal_size_mm']:.0f}M\nResolving...")
TARGET_RESOLVED = resolve_target_financials(TARGET, df)
print(f"\nEBITDA margin : {TARGET_RESOLVED.get('ebitda_margin_pct', 'N/A')}"
      if not isinstance(TARGET_RESOLVED.get('ebitda_margin_pct'), float)
      else f"\nEBITDA margin : {TARGET_RESOLVED['ebitda_margin_pct']:.1f}%")

Target   : Example Target Co.
Sector   : Health IT  |  Geography: Northeast  |  Ownership: Private
Size     : $200M
Resolving...
  'strong' ebitda_margin_pct for 'Health IT' → 26.0% (p75 of sector)

EBITDA margin : 26.0%


## 2. Scoring Engine

Three steps run in sequence:

1. **Build acquirer profiles** — aggregate each buyer's historical deal statistics
2. **Score each dimension** — map distance from the target profile to a 0–100 score
3. **Rank all acquirers** — compute the weighted composite and return a sorted table

In [4]:
# ── Step 1: Build a statistical profile for each acquirer ─────────────────────

def build_acquirer_profiles(df):
    profiles = {}
    for acquirer, g in df.groupby('acquirer'):
        sizes = g['deal_size_mm'].dropna()
        margins = g['ebitda_margin_pct'].dropna()
        profiles[acquirer] = {
            'acquirer_type':  g['acquirer_type'].mode()[0],
            'num_deals':      len(g),
            'last_deal_year': int(g['deal_year'].max()),
            'sectors':        g['sector'].value_counts().to_dict(),
            'geographies':    g['geography'].value_counts().to_dict(),
            'ownerships':     g['target_ownership_pre'].value_counts().to_dict(),
            'size_mean':      sizes.mean(),
            'size_std':       sizes.std() if len(sizes) > 1 else sizes.mean() * 0.40,
            'margin_mean':    margins.mean(),
            'margin_std':     margins.std() if len(margins) > 1 else margins.mean() * 0.35,
        }
    return profiles


# ── Step 2: Per-dimension scoring ─────────────────────────────────────────────

def _zscore_score(value, mean, std):
    """Map z-score distance to 0–100 (closer = higher)."""
    std = max(std, abs(mean) * 0.10, 1.0)
    z = abs(value - mean) / std
    if z <= 0.5: return 100.0
    if z <= 1.0: return 80.0
    if z <= 1.5: return 55.0
    if z <= 2.0: return 30.0
    return max(0.0, 30.0 - 12.0 * (z - 2.0))


def score_acquirer(acq, target):
    n = acq['num_deals']
    na_dims = target.get('_na_dims', [])
    s = {}

    s['sector'] = 100.0 * acq['sectors'].get(target['sector'], 0) / n

    s['deal_size'] = _zscore_score(target['deal_size_mm'], acq['size_mean'], acq['size_std'])

    geo_count = acq['geographies'].get(target['geography'], 0)
    if target['geography'] not in ('National', 'Multi-Regional'):
        geo_count += 0.5 * acq['geographies'].get('National', 0)
        geo_count += 0.5 * acq['geographies'].get('Multi-Regional', 0)
    s['geography'] = min(100.0, 100.0 * geo_count / n)

    if 'ebitda_margin' not in na_dims:
        s['ebitda_margin'] = _zscore_score(
            target['ebitda_margin_pct'], acq['margin_mean'], acq['margin_std'])
    else:
        s['ebitda_margin'] = 0.0

    s['ownership'] = 100.0 * acq['ownerships'].get(target['ownership'], 0) / n

    years_ago = 2025 - acq['last_deal_year']
    s['recency'] = max(0.0, 100.0 * (0.80 ** years_ago))

    return s


# ── Step 3: Full pipeline ─────────────────────────────────────────────────────

def run_screener(closed_df, target, top_n=20):
    profiles = build_acquirer_profiles(closed_df)
    w = target['weights']
    dims = list(w.keys())
    active_dims = [d for d in dims if w[d] > 0]

    rows = []
    for acquirer, acq in profiles.items():
        if (target['preferred_acquirer_type'] and
                acq['acquirer_type'] != target['preferred_acquirer_type']):
            continue
        raw = score_acquirer(acq, target)
        composite = sum(w[d] * raw[d] for d in active_dims)
        grade = 'A' if composite >= 80 else ('B' if composite >= 65 else
                ('C' if composite >= 50 else 'D'))
        rows.append({
            'acquirer':       acquirer,
            'acquirer_type':  acq['acquirer_type'],
            'num_deals':      acq['num_deals'],
            'last_deal_year': acq['last_deal_year'],
            **{f'score_{d}': round(raw[d], 1) for d in active_dims},
            'composite_score': round(composite, 1),
            'grade': grade,
        })

    out = (pd.DataFrame(rows)
           .sort_values('composite_score', ascending=False)
           .reset_index(drop=True))
    out.index = out.index + 1
    out.index.name = 'rank'

    score_cols = [f'score_{d}' for d in active_dims]
    disp_cols = ['acquirer', 'acquirer_type', 'num_deals', 'last_deal_year'] + \
                score_cols + ['composite_score', 'grade']
    fmt = {c: '{:.1f}' for c in score_cols + ['composite_score']}

    na_note = (f"  (excluded: {target.get('_na_dims')})" if target.get('_na_dims') else "")
    print(f"Target: {target['name']}  |  {len(out)} acquirers scored{na_note}")
    display(
        out[disp_cols].head(top_n)
        .style
        .background_gradient(subset=['composite_score'], cmap='RdYlGn', vmin=0, vmax=100)
        .background_gradient(subset=score_cols, cmap='Blues', vmin=0, vmax=100)
        .format(fmt)
        .set_caption(f"Most likely buyers for {target['name']} — ranked by composite score")
    )
    return out


print('Engine ready.')

Engine ready.


## 2. Results

In [5]:
results = run_screener(closed, TARGET_RESOLVED, top_n=20)

Target: Example Target Co.  |  97 acquirers scored


,acquirer,acquirer_type,num_deals,last_deal_year,score_sector,score_deal_size,score_geography,score_ebitda_margin,score_ownership,score_recency,composite_score,grade
rank,,,,,,,,,,,,
1,Netsmart Technologies,Strategic,1,2016,100.0,80.0,50.0,100.0,100.0,13.4,83.2,A
2,Cerner (Oracle),Strategic,2,2024,100.0,100.0,50.0,100.0,0.0,80.0,81.5,A
3,Aledade,Strategic,1,2020,100.0,80.0,50.0,80.0,100.0,32.8,81.1,A
4,Change Healthcare,Strategic,2,2023,100.0,80.0,75.0,55.0,50.0,64.0,77.7,B
5,Evolent Health,Strategic,3,2022,100.0,100.0,0.0,100.0,33.3,51.2,75.9,B
6,Arcadian Telepsychiatry,Strategic,1,2023,100.0,0.0,100.0,80.0,100.0,64.0,70.2,B
7,Epic Systems,Strategic,3,2022,100.0,80.0,16.7,80.0,0.0,51.2,67.1,B
8,Veeva Systems,Strategic,1,2023,100.0,80.0,0.0,80.0,0.0,64.0,65.2,B
9,Ciox Health,Strategic,4,2020,100.0,100.0,25.0,30.0,0.0,32.8,64.9,C


## 3. M&A Feasibility Pipeline

Takes the **top-scored acquirers**, runs parallel web searches for each, then
calls a local Ollama model to produce a one-page feasibility assessment.

Steps in this section:
- **3a** — LLM & pipeline configuration ← *edit here to change model or top-N*
- **3b** — Imports, output schema & web-search tool
- **3c** — Agent loop & retry logic
- **3d** — Prompt templates & message builder
- **3e** — Run the pipeline
- **3f** — Render & export PDF reports

### 3a. LLM & Pipeline Config
Change `OLLAMA_MODEL` to switch models. `TOP_N_PIPELINE` controls how many
acquirers are assessed.

In [6]:
# ── Config ────────────────────────────────────────────────────────────────────
OLLAMA_MODEL    = "gpt-oss:20b-cloud"
TOP_N_PIPELINE  = 10
MAX_TOOL_ROUNDS = 5
MAX_RETRIES     = 5


### 3b. Imports, Output Schema & Web-Search Tool

In [7]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from ollama import chat
from pydantic import BaseModel, Field, ValidationError
from typing import Literal
from ddgs import DDGS


# ── Output schema ─────────────────────────────────────────────────────────────
class Assessment(BaseModel):
    reasoning_trace:      str = Field(..., description="Internal chain-of-thought before final answer")
    company_overview:     str = Field(..., description="Acquirer size, strategic priorities, M&A patterns")
    strategic_fit:        str = Field(..., description="Why this target aligns with acquirer strategy")
    precedent_activity:   str = Field(..., description="2-3 relevant prior transactions with deal size and multiples")
    valuation_context:    str = Field(..., description="EV/EBITDA and EV/Revenue ranges, valuation estimate")
    risk_flags:           str = Field(..., description="At least 2 material risks")
    conviction_level:     Literal["High", "Medium", "Low"]
    conviction_rationale: str = Field(..., description="1-2 sentences supporting the conviction rating")

# ── Tool definition ───────────────────────────────────────────────────────────
WEB_SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search the web for up-to-date information about a company or topic.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"}
            },
            "required": ["query"]
        }
    }
}

def web_search(query, max_results=3):
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        return " | ".join(r["body"] for r in results) if results else "No results."
    except Exception as e:
        return f"Search error: {e}"

TOOLS = {"web_search": web_search}



### 3c. Agent Loop & Retry Logic

In [8]:
def _execute_tool(tc):
    try:
        fn_name = tc.function.name
        fn_args = tc.function.arguments or {}
        if isinstance(fn_args, str):
            fn_args = json.loads(fn_args)
        if fn_name not in TOOLS:
            return f"Unknown tool: {fn_name}"
        return TOOLS[fn_name](**fn_args)
    except json.JSONDecodeError as e:
        return f"Tool argument parse error: {e}"
    except Exception as e:
        return f"Tool execution error ({fn_name}): {e}"

# ── Agent loop ────────────────────────────────────────────────────────────────
def run_agent(messages, model=OLLAMA_MODEL, max_rounds=MAX_TOOL_ROUNDS):
    for _ in range(max_rounds):
        response = chat(
            model=model,
            messages=messages,
            tools=[WEB_SEARCH_TOOL],
            options={"seed": 42, "temperature": 0}
        )
        msg = response.message
        messages.append(msg)

        if not msg.tool_calls:
            return msg.content.strip(), messages

        with ThreadPoolExecutor(max_workers=len(msg.tool_calls)) as ex:
            futures = {ex.submit(_execute_tool, tc): tc for tc in msg.tool_calls}
            for fut in as_completed(futures):
                messages.append({"role": "tool", "content": fut.result()})

    return "Max tool rounds reached.", messages

# ── Retry loop: full agent once, then JSON-fix only (no tool re-runs) ─────────
def _clean_json(raw):
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:])
    if cleaned.endswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[:-1])
    return cleaned.strip()

def run_agent_with_retry(messages, model=OLLAMA_MODEL, max_retries=MAX_RETRIES):
    raw, messages = run_agent(messages, model)

    for attempt in range(1, max_retries + 1):
        try:
            return Assessment(**json.loads(_clean_json(raw))), attempt
        except (json.JSONDecodeError, ValidationError) as e:
            if attempt == max_retries:
                raise
            print(f"  [retry {attempt}/{max_retries}] {e}")
            fix_messages = messages + [{
                "role": "user",
                "content": (
                    f"Fix this JSON error and return corrected JSON only — "
                    f"no prose, no code fences:\n{e}\n\n"
                    f"Schema:\n{json.dumps(Assessment.model_json_schema(), indent=2)}"
                )
            }]
            response = chat(model=model, messages=fix_messages,
                            options={"temperature": 0})
            raw = response.message.content

# ── Prompts ───────────────────────────────────────────────────────────────────


### 3d. Prompt Templates & Message Builder

In [9]:
SYSTEM_PROMPT = """You are an Investment Banking VP providing a concise, high-conviction M&A feasibility assessment.

## Your Task
You will receive a target company and an acquirer's deal history. Your job is to:
1. Call `web_search` for current information on the acquirer — issue ALL searches in ONE parallel batch before reasoning.
2. Synthesize web results with the dataset to produce a structured JSON assessment.

## PDF Rendering Rules
Your output is rendered as a one-page PDF. Each JSON field is converted from Markdown to HTML:
- When using bullet points, use `- `
- Use `**bold**` for key figures (deal sizes, multiples, percentages)
- Do NOT use headers (`##`) or numbered lists inside any field — the PDF template provides structure
- `precedent_activity` and `valuation_context` render side-by-side — keep them the same number of bullets

## Writing Style
- Write every bullet as a full sentence with a subject and verb
- Avoid noun-phrase fragments (e.g. ❌ "Strong sector alignment" → ✅ "The target's software focus aligns with the acquirer's stated platform consolidation strategy.")
- Be specific — use numbers, names, and dates wherever available

## Section Definitions
- `reasoning_trace`: Your internal chain-of-thought — NOT rendered in PDF, free-form reasoning allowed
- `company_overview`: Who they are: size, strategic priorities, recent M&A activity implied by the dataset
- `strategic_fit`: Why this specific target is a logical acquisition — sector alignment, capability gap, geographic expansion, or platform build thesis
- `precedent_activity`: Relevant prior transactions from the dataset: — each bullet must include: company name, year, sector, deal size ($M), and EV multiple
- `valuation_context`: EV/EBITDA and EV/Revenue ranges from comparable closed deals in the dataset; close with one sentence on implied valuation range for this target
- `risk_flags`: At least 2 risks — format each as: `- **[Risk Name]:** One sentence explaining why it is material for this deal`
- `conviction_level`: Exactly one of: "High", "Medium", or "Low" — no other values accepted
- `conviction_rationale`: 1–2 sentences tying the rating directly to the fit score, deal history, and any web-sourced context

## Output Schema
Return a single JSON object conforming exactly to:

    class Assessment(BaseModel):
        reasoning_trace:      str   # Internal CoT — not rendered
        company_overview:     str   # Acquirer size, strategy, M&A pattern
        strategic_fit:        str   # Why target aligns with acquirer
        precedent_activity:   str   # 2-3 prior deals with size + multiples
        valuation_context:    str   # Comparable multiples + implied range
        risk_flags:           str   # Exactly 2 named, explained risks
        conviction_level:     Literal["High", "Medium", "Low"]
        conviction_rationale: str   # 1-2 sentences on conviction rating
"""


def build_user_message(target: dict, acquirer_row, acquirer_deals) -> str:
    """
    Build the user-turn message for the M&A assessment pipeline.

    Args:
        target:        Dict of target company characteristics
        acquirer_row:  Series/dict row from the acquirer summary DataFrame
        acquirer_deals: DataFrame of the acquirer's historical deals
    Returns:
        Formatted string for the user message turn
    """

    # --- Target block ---
    ebitda = target.get("ebitda_margin_pct", "N/A")
    ebitda_str = f"{ebitda:.1f}%" if isinstance(ebitda, (int, float)) else ebitda

    target_block = (
        f"Name:          {target['name']}\n"
        f"Sector:        {target['sector']}\n"
        f"Geography:     {target['geography']}\n"
        f"Ownership:     {target['ownership']}\n"
        f"Deal Size:     ${target['deal_size_mm']:.0f}M\n"
        f"EBITDA Margin: {ebitda_str}"
    )

    # --- Acquirer block ---
    acquirer_block = (
        f"Name:              {acquirer_row['acquirer']}\n"
        f"Type:              {acquirer_row['acquirer_type']}\n"
        f"Closed Deals:      {int(acquirer_row['num_deals'])} (in dataset)\n"
        f"Last Active:       {int(acquirer_row['last_deal_year'])}\n"
        f"Fit Score:         {acquirer_row['composite_score']:.1f} / 100"
    )

    # --- Deal history block — CSV is cleaner and more token-efficient than to_string() ---
    cols = [
        'target_company', 'deal_year', 'sector', 'deal_size_mm',
        'ebitda_margin_pct', 'deal_type', 'ev_ebitda_multiple', 'ev_revenue_multiple'
    ]
    deals_csv = acquirer_deals[cols].to_csv(index=False)

    acquirer_name = acquirer_row['acquirer']

    return (
        f"## Step 1 — Search\n"
        f"Call web_search for **{acquirer_name}** now. "
        f"Retrieve: current revenue/EBITDA, recent M&A announcements, stated strategic priorities, "
        f"and any news relevant to this deal. Issue all queries in ONE parallel batch.\n\n"

        f"## Step 2 — Assess\n"
        f"Using your search results and the data below, return the JSON assessment.\n\n"

        f"### Target\n{target_block}\n\n"

        f"### Acquirer\n{acquirer_block}\n\n"

        f"### Acquirer Deal History (dataset)\n"
        f"```csv\n{deals_csv}```\n\n"

        f"Return a single JSON object matching the Assessment schema. "
        f"Do not include any text outside the JSON block."
    )


print("Pipeline ready  |  TOP_N_PIPELINE =", TOP_N_PIPELINE)

Pipeline ready  |  TOP_N_PIPELINE = 10


In [10]:
import time

def run_feasibility_pipeline(results, closed_df, target, top_n=TOP_N_PIPELINE,
                              stagger_secs=2.5):
    """
    stagger_secs: delay between each agent submission to avoid
                  overwhelming the Ollama server with concurrent requests (429s).
    """
    top = results.head(top_n)

    def assess_acquirer(row):
        acquirer_deals = closed_df[closed_df['acquirer'] == row['acquirer']]
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_message(target, row, acquirer_deals)},
        ]
        assessment, attempts = run_agent_with_retry(messages)
        return row['acquirer'], assessment, attempts

    assessments = {}
    with ThreadPoolExecutor(max_workers=top_n) as ex:
        futures = {}
        for i, (_, row) in enumerate(top.iterrows()):
            if i > 0:
                time.sleep(stagger_secs)
            futures[ex.submit(assess_acquirer, row)] = row['acquirer']

        for fut in as_completed(futures):
            name = futures[fut]
            try:
                _, assessment, attempts = fut.result()
                assessments[name] = assessment
                retry_note = f"  ({attempts} attempt{'s' if attempts > 1 else ''})"
                print(f"  ✓ {name}{retry_note}")
            except Exception as e:
                assessments[name] = None
                print(f"  ✗ {name}  ({e})")

    ranked = [r['acquirer'] for _, r in top.iterrows()]
    return {name: assessments[name] for name in ranked if name in assessments}


print("Running pipeline...")
assessments = run_feasibility_pipeline(results, closed, TARGET_RESOLVED)

Running pipeline...
  [retry 1/5] Invalid \escape: line 2 column 163 (char 164)
  ✓ Cerner (Oracle)  (1 attempt)
  ✓ Aledade  (1 attempt)
  ✓ Netsmart Technologies  (2 attempts)
  [retry 1/5] 8 validation errors for Assessment
reasoning_trace
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.9/v/missing
company_overview
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.9/v/missing
strategic_fit
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.9/v/missing
precedent_activity
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.9/v/missing
valuation_context
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.de

### 3f. Render & Export PDFs

In [11]:
from jinja2 import Environment
from weasyprint import HTML as WeasyprintHTML
import markdown as md_lib
import os

jinja_env = Environment()
jinja_env.filters["markdown"] = \
    lambda text: md_lib.markdown(text or "", extensions=["tables", 'nl2br'])

TEMPLATE = jinja_env.from_string("""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  body { font-family: 'Helvetica Neue', Arial, sans-serif; font-size: 11px;
         color: #1a1a1a; margin: 0; padding: 30px 40px; }
  h1   { font-size: 16px; font-weight: bold; border-bottom: 2px solid #1a1a1a;
         padding-bottom: 6px; margin-bottom: 4px; }
  .meta { font-size: 10px; color: #555; margin-bottom: 18px; }
  h2   { font-size: 12px; font-weight: bold; color: #1a3a5c;
         margin: 14px 0 4px; text-transform: uppercase; letter-spacing: 0.04em; }
  p, li { line-height: 1.5; margin: 2px 0; }
  ul   { margin: 4px 0 4px 16px; padding: 0; }
  .conviction { display: inline-block; padding: 2px 10px; border-radius: 3px;
                font-weight: bold; font-size: 11px; }
  .High   { background: #d4edda; color: #155724; }
  .Medium { background: #fff3cd; color: #856404; }
  .Low    { background: #f8d7da; color: #721c24; }
  .section { margin-bottom: 10px; }
  hr { border: none; border-top: 1px solid #ddd; margin: 8px 0; }
</style>
</head>
<body>
  <h1>M&A Feasibility Assessment: {{ acquirer }}</h1>
  <div class="meta">
    Target: {{ target_name }} &nbsp;|&nbsp;
    Sector: {{ sector }} &nbsp;|&nbsp;
    Deal Size: ${{ deal_size }}M &nbsp;|&nbsp;
    Fit Score: {{ score }}/100 &nbsp;|&nbsp;
    Rank: #{{ rank }}
  </div>

  <div class="section">
    <h2>1. Company Overview</h2>
    {{ company_overview | markdown }}
  </div><hr>

  <div class="section">
    <h2>2. Strategic Fit Thesis</h2>
    {{ strategic_fit | markdown }}
  </div><hr>

  <div class="section">
    <h2>3. Precedent Activity</h2>
    {{ precedent_activity | markdown }}
  </div><hr>

  <div class="section">
    <h2>4. Valuation Context</h2>
    {{ valuation_context | markdown }}
  </div><hr>
                                 
  <div class="section">
    <h2>5. Risk Flags</h2>
    {{ risk_flags | markdown }}
  </div><hr>

  <div class="section">
    <h2>6. Conviction Level &nbsp;
      <span class="conviction {{ conviction_level }}">{{ conviction_level }}</span>
    </h2>
    <p>{{ conviction_rationale }}</p>
  </div>
</body>
</html>
""")

OUTPUT_DIR = "assessments"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def render_pdf(rank, acquirer_name, assessment, target, acquirer_row):
    html_str = TEMPLATE.render(
        rank=rank,
        acquirer=acquirer_name,
        target_name=target["name"],
        sector=target["sector"],
        deal_size=int(target["deal_size_mm"]),
        score=acquirer_row["composite_score"],
        company_overview=assessment.company_overview,
        strategic_fit=assessment.strategic_fit,
        precedent_activity=assessment.precedent_activity,
        valuation_context=assessment.valuation_context,
        risk_flags=assessment.risk_flags,
        conviction_level=assessment.conviction_level,
        conviction_rationale=assessment.conviction_rationale,
    )
    safe_name = acquirer_name.replace("/", "-").replace(" ", "_")
    pdf_path  = os.path.join(OUTPUT_DIR, f"{rank:02d}_{safe_name}.pdf")
    WeasyprintHTML(string=html_str).write_pdf(pdf_path)
    return pdf_path

top = results.head(TOP_N_PIPELINE)
acquirer_rows = {r["acquirer"]: r for _, r in top.iterrows()}

for rank, (name, a) in enumerate(assessments.items(), 1):
    if a is None:
        print(f"  ✗ #{rank} {name} — skipped (failed validation)")
        continue
    path = render_pdf(rank, name, a, TARGET_RESOLVED, acquirer_rows[name])
    print(f"  ✓ #{rank} {name} → {path}")

print(f"\nPDFs written to ./{OUTPUT_DIR}/")

  ✓ #1 Netsmart Technologies → assessments/01_Netsmart_Technologies.pdf
  ✓ #2 Cerner (Oracle) → assessments/02_Cerner_(Oracle).pdf
  ✓ #3 Aledade → assessments/03_Aledade.pdf
  ✓ #4 Change Healthcare → assessments/04_Change_Healthcare.pdf
  ✓ #5 Evolent Health → assessments/05_Evolent_Health.pdf
  ✓ #6 Arcadian Telepsychiatry → assessments/06_Arcadian_Telepsychiatry.pdf
  ✓ #7 Epic Systems → assessments/07_Epic_Systems.pdf
  ✓ #8 Veeva Systems → assessments/08_Veeva_Systems.pdf
  ✓ #9 Ciox Health → assessments/09_Ciox_Health.pdf
  ✓ #10 Privia Health → assessments/10_Privia_Health.pdf

PDFs written to ./assessments/
